# **IoT 보안 데이터 조회 시스템 (Text2SQL)**

## 프로젝트 목표
1. IoT 보안·하드웨어·사고 대응 CSV 데이터를 Supabase에 적재
2. SQL 쿼리로 데이터 조회 테스트
3. 자연어 기반 IoT 보안 데이터 조회 시스템 구현 및 테스트

## 구현 단계
- 환경 설정 확인
- CSV 파일 확인 및 탐색
- Supabase 연결
- CSV 데이터 업로드
- SQL 쿼리 테스트
- IoT 보안 Text2SQL 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [144]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Supabase 설정 확인
if os.environ.get("SUPABASE_DB_URL"):
    print("✓ Supabase DB URL이 설정되었습니다.")
else:
    print("✗ Supabase DB URL이 필요합니다.")
    print("  .env 파일에 SUPABASE_DB_URL을 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Supabase DB URL이 설정되었습니다.


## 1. CSV 파일 확인 및 탐색

IoT 보안·하드웨어·사고 대응 CSV 파일을 불러옵니다.

**중요:** 여러 CSV 파일을 사용하는 경우, 테이블 간 관계(Foreign Key)를 고려하여 업로드 순서를 결정하세요.
- 부모 테이블 → 자식 테이블 순서로 업로드

In [145]:
import pandas as pd

# IoT 보안 데이터 CSV 경로
# 여러 파일이 있다면 dict 형태로 구성
# 예시:
# csv_files = {
#     "table1": "../datasets/your_table1.csv",
#     "table2": "../datasets/your_table2.csv"
# }

csv_files = {
    "iot_common_guidelines": "../datasets/1_parent_iot_common_guidelines.csv",
    "home_iot_controls": "../datasets/2_child_home_iot_controls.csv",
    "stm32_debug_topics": "../datasets/3_child_stm32_debug_topics.csv",
    "esp32h2_features": "../datasets/4_child_esp32h2_features.csv",
    "iot_security_agencies_parent": "../datasets/iot_security_agencies_parent.csv",
    "iot_security_incidents_child": "../datasets/iot_security_incidents_child.csv"
}


# CSV 파일 로드 및 확인
dataframes = {}

for table_name, file_path in csv_files.items():
    try:
        df = pd.read_csv(file_path)
        dataframes[table_name] = df

        print("=" * 80)
        print(f"📋 {table_name} 테이블")
        print("=" * 80)
        print(f"\n행 수: {len(df)}")
        print(f"컬럼: {list(df.columns)}")
        print(f"\n첫 5개 행:")
        print(df.head())
        print(f"\n데이터 타입:")
        print(df.dtypes)
        print("\n")

    except Exception as e:
        print(f"✗ {table_name} 로드 실패: {e}\n")

print(f"\n✓ 총 {len(dataframes)}개의 테이블 로드 완료")

📋 iot_common_guidelines 테이블

행 수: 15
컬럼: ['guideline_id', 'principle_id', 'principle_name', 'lifecycle_stage', 'guideline_code', 'guideline_name', 'page_start', 'domain', 'description', 'design_focus', 'security_focus', 'easy_explanation']

첫 5개 행:
   guideline_id  principle_id                     principle_name  \
0             1             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
1             2             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
2             3             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
3             4             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   
4             5             1  정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계   

  lifecycle_stage guideline_code                guideline_name  page_start  \
0           설계·개발            G01     IoT 장치 특성을 고려한 보안 서비스 경량화          20   
1           설계·개발            G02  접근권한 관리·인증·종단간 통신 보안·데이터 암호화          31   
2           설계·개발            G03         소프트웨어·하드웨어 보안기술 적용 검토          33   
3           설계·개발            G

## 2. 데이터 탐색 및 통계

IoT 보안 데이터의 컬럼, 분포와 주요 항목을 확인합니다.

In [146]:
for table_name, df in dataframes.items():
    print(f"\n{'='*80}")
    print(f"📊 {table_name} 통계")
    print("="*80)

    # 기본 통계
    print("\n[기본 정보]")
    df.info()

    # 결측치 확인
    print("\n[결측치]")
    null_counts = df.isnull().sum()

    if null_counts.sum() > 0:
        print(null_counts[null_counts > 0])
    else:
        print("결측치 없음")

    # 고유값 개수
    print("\n[고유값 개수]")
    for col in df.columns:
        print(f"{col}: {df[col].nunique()}개")

    # 카테고리별 데이터 분포
    print("\n[카테고리 분포]")
    for col in df.select_dtypes(include=['object']).columns:
        print(f"\n[{col}]")
        print(df[col].value_counts())


📊 iot_common_guidelines 통계

[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   guideline_id      15 non-null     int64
 1   principle_id      15 non-null     int64
 2   principle_name    15 non-null     str  
 3   lifecycle_stage   15 non-null     str  
 4   guideline_code    15 non-null     str  
 5   guideline_name    15 non-null     str  
 6   page_start        15 non-null     int64
 7   domain            15 non-null     str  
 8   description       15 non-null     str  
 9   design_focus      15 non-null     str  
 10  security_focus    15 non-null     str  
 11  easy_explanation  15 non-null     str  
dtypes: int64(3), str(9)
memory usage: 8.7 KB

[결측치]
결측치 없음

[고유값 개수]
guideline_id: 15개
principle_id: 7개
principle_name: 7개
lifecycle_stage: 3개
guideline_code: 15개
guideline_name: 15개
page_start: 15개
domain: 15개
description: 15개
design_focus

easy_explanation: 15개

[카테고리 분포]

[principle_name]
principle_name
정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계      5
안전한 소프트웨어 및 하드웨어 개발기술 적용 및 검증          3
IoT 제품·서비스 취약점 보안패치 및 업데이트 지속 이행       2
IoT 침해사고 대응체계 및 책임추적성 확보 방안 마련         2
안전한 초기 보안설정 방안 제공                      1
안전한 설치를 위한 보안 프로토콜 준수 및 안전한 파라미터 설정    1
안전한 운영·관리를 위한 정보보호 및 프라이버시 관리체계 마련     1
Name: count, dtype: int64

[lifecycle_stage]
lifecycle_stage
설계·개발       8
운영·관리·폐기    5
배포·설치·구성    2
Name: count, dtype: int64

[guideline_code]
guideline_code
G01    1
G02    1
G03    1
G04    1
G05    1
G06    1
G07    1
G08    1
G09    1
G10    1
G11    1
G12    1
G13    1
G14    1
G15    1
Name: count, dtype: int64

[guideline_name]
guideline_name
IoT 장치 특성을 고려한 보안 서비스 경량화       1
접근권한 관리·인증·종단간 통신 보안·데이터 암호화    1
소프트웨어·하드웨어 보안기술 적용 검토           1
민감정보 보호                         1
민감정보 운영정책 투명성 보장                1
시큐어코딩 적용                        1
소프트웨어 취약점 점검 및 보안패치 방안 구현       1
다양한 하드웨어 보안기법 적용                1
Secure by Default 적

C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_4952\672413444.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:
C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_4952\672413444.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_gu

<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   esp32_feature_id       26 non-null     int64
 1   parent_guideline_id    26 non-null     int64
 2   feature_name           26 non-null     str  
 3   chapter_no             26 non-null     int64
 4   page_start             26 non-null     int64
 5   feature_category       26 non-null     str  
 6   module_or_interface    26 non-null     str  
 7   hardware_or_security   26 non-null     str  
 8   design_use             26 non-null     str  
 9   description            26 non-null     str  
 10  hardware_design_point  26 non-null     str  
 11  security_purpose       26 non-null     str  
 12  easy_explanation       26 non-null     str  
dtypes: int64(4), str(9)
memory usage: 13.3 KB

[결측치]
결측치 없음

[고유값 개수]
esp32_feature_id: 26개
parent_guideline_id: 7개
feature_name: 26개
chapter_no: 22개
page_st

C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_4952\672413444.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:
C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_4952\672413444.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_gu

## 3. Supabase PostgreSQL 연결

In [147]:
from langchain_community.utilities import SQLDatabase

supabase_db_url = os.getenv("SUPABASE_DB_URL")

if not supabase_db_url:
    raise Exception("SUPABASE_DB_URL이 설정되지 않았습니다. .env 파일을 확인하세요.")

print("Supabase PostgreSQL 연결 중...\n")

try:
    # LangChain SQLDatabase로 PostgreSQL 연결
    db = SQLDatabase.from_uri(supabase_db_url)

    print("✓ PostgreSQL 연결 성공!\n")
    print("현재 테이블 목록:")
    tables = db.get_usable_table_names()
    print(tables)

except Exception as e:
    print(f"✗ 연결 실패: {e}")
    raise

Supabase PostgreSQL 연결 중...



✓ PostgreSQL 연결 성공!

현재 테이블 목록:
['1_parent_iot_common_guidelines', '2_child_home_iot_controls', '3_child_stm32_debug_topics', '4_child_esp32h2_features', 'iot_security_agencies_parents', 'iot_security_incidents_child', 'security_agencies', 'security_incidents']


## 4. Supabase에 CSV 데이터 업로드

### Supabase 대시보드 (GUI)
1. Supabase 대시보드 접속
2. Table Editor → Import data from CSV
3. **반드시 테이블 관계 순서대로 업로드** (부모 → 자식)
4. Foreign Key 에러 발생 시 순서를 재확인

## 5. 업로드 확인 및 스키마 탐색

In [148]:
# 데이터베이스 다시 연결 (업로드 후 스키마 갱신)
db = SQLDatabase.from_uri(supabase_db_url)

print("=== 데이터베이스 스키마 ===")
print(db.table_info)

print("\n" + "="*80 + "\n")

# 각 테이블의 샘플 데이터
for table in db.get_usable_table_names():
    print(f"{table} 테이블 샘플:")
    try:
        result = db.run(f"SELECT * FROM {table} LIMIT 3")
        print(result)
    except Exception as e:
        print(f"조회 실패: {e}")
    print()

=== 데이터베이스 스키마 ===

CREATE TABLE "1_parent_iot_common_guidelines" (
	guideline_id BIGINT, 
	principle_id BIGINT, 
	principle_name TEXT, 
	lifecycle_stage TEXT, 
	guideline_code TEXT, 
	guideline_name TEXT, 
	page_start BIGINT, 
	domain TEXT, 
	description TEXT, 
	design_focus TEXT, 
	security_focus TEXT, 
	easy_explanation TEXT
)

/*
3 rows from 1_parent_iot_common_guidelines table:
guideline_id	principle_id	principle_name	lifecycle_stage	guideline_code	guideline_name	page_start	domain	description	design_focus	security_focus	easy_explanation
1	1	정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계	설계·개발	G01	IoT 장치 특성을 고려한 보안 서비스 경량화	20	하드웨어_설계	프로세서 성능, 메모리, 입출력장치, 소비전력 등 장치 자원 수준을 고려하여 필요한 보안 기능을 경량화해 구현	MCU 성능, 메모리, 입출력장치, 소비전력 등 기기 자원을 먼저 확인	기기 성능을 넘지 않는 범위에서 필요한 보안 기능을 적용	기기의 성능과 전력에 맞춰 하드웨어와 보안 기능의 크기를 정하는 기준
2	1	정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계	설계·개발	G02	접근권한 관리·인증·종단간 통신 보안·데이터 암호화	31	인증_통신_암호화	IoT 서비스 환경에 맞는 접근권한, 인증, 통신 보호, 데이터 암호화 방안을 제공	기기와 외부 장치·서버가 연결되는 통신 경로를 확인	허가된 사용자·장치만 접근하게 하고 통신과 데이터를

## 6. SQL 쿼리 테스트

IoT 보안 데이터를 조회하는 읽기 전용 SQL을 테스트합니다.

In [149]:
# TODO: 기본 조회 쿼리 작성
# ESP32-H2의 하드웨어·보안 기능 통합 조회
# 부모: 1_parent_iot_common_guidelines
# 자식: 4_child_esp32h2_features

query = """
SELECT
    e.feature_name,
    e.feature_category,
    e.module_or_interface,
    e.hardware_or_security,
    e.design_use,
    e.page_start,
    g.guideline_name
FROM "4_child_esp32h2_features" AS e
INNER JOIN "1_parent_iot_common_guidelines" AS g
    ON e.parent_guideline_id = g.guideline_id
WHERE e.hardware_or_security IN ('하드웨어', '보안', '통합')
ORDER BY
    CASE e.hardware_or_security
        WHEN '하드웨어' THEN 1
        WHEN '통합' THEN 2
        WHEN '보안' THEN 3
        ELSE 4
    END,
    e.page_start;
"""

print("실행 쿼리:")
print(query)
print("\n결과:")

try:
    result = db.run(query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT
    e.feature_name,
    e.feature_category,
    e.module_or_interface,
    e.hardware_or_security,
    e.design_use,
    e.page_start,
    g.guideline_name
FROM "4_child_esp32h2_features" AS e
INNER JOIN "1_parent_iot_common_guidelines" AS g
    ON e.parent_guideline_id = g.guideline_id
WHERE e.hardware_or_security IN ('하드웨어', '보안', '통합')
ORDER BY
    CASE e.hardware_or_security
        WHEN '하드웨어' THEN 1
        WHEN '통합' THEN 2
        WHEN '보안' THEN 3
        ELSE 4
    END,
    e.page_start;


결과:
[('System and Memory', '메모리 구조', 'ROM / HP SRAM / LP SRAM / External Flash', '하드웨어', '메모리 설계', 142, 'IoT 장치 특성을 고려한 보안 서비스 경량화'), ('IO MUX and GPIO Matrix', '입출력 설계', 'GPIO / IO MUX', '하드웨어', '핀 기능·신호 라우팅 설계', 217, 'IoT 장치 특성을 고려한 보안 서비스 경량화'), ('GPIO Hysteresis', '신호 안정성', 'GPIO', '하드웨어', '입력 노이즈 대응', 227, 'IoT 장치 특성을 고려한 보안 서비스 경량화'), ('GPIO Power Supply Management', '전원·저전력 설계', 'VDDPST1 / VDDPST2 / VDDA_PMU/VBAT', '하드웨어', 'GPIO 전원 도메인·Sleep Wake 설계', 228, 'IoT 장치 특성을 고려

In [150]:
# TODO: JOIN 쿼리 작성
# 부모 테이블과 자식 3개 테이블을 연결하여 하드웨어·보안 데이터 통합 조회

join_query = """
SELECT
    g.guideline_name,
    '홈가전 IoT 보안가이드' AS source_type,
    h.control_name AS item_name,
    h.hardware_or_security AS item_type,
    h.page_start
FROM "2_child_home_iot_controls" h
INNER JOIN "1_parent_iot_common_guidelines" g
    ON h.parent_guideline_id = g.guideline_id

UNION ALL

SELECT
    g.guideline_name,
    'STM32 디버깅 가이드' AS source_type,
    s.topic_name AS item_name,
    s.hardware_or_security AS item_type,
    s.page_start
FROM "3_child_stm32_debug_topics" s
INNER JOIN "1_parent_iot_common_guidelines" g
    ON s.parent_guideline_id = g.guideline_id

UNION ALL

SELECT
    g.guideline_name,
    'ESP32-H2 기술 매뉴얼' AS source_type,
    e.feature_name AS item_name,
    e.hardware_or_security AS item_type,
    e.page_start
FROM "4_child_esp32h2_features" e
INNER JOIN "1_parent_iot_common_guidelines" g
    ON e.parent_guideline_id = g.guideline_id

ORDER BY guideline_name, source_type, page_start;
"""

print("실행 쿼리:")
print(join_query)
print("\n결과:")

try:
    result = db.run(join_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT
    g.guideline_name,
    '홈가전 IoT 보안가이드' AS source_type,
    h.control_name AS item_name,
    h.hardware_or_security AS item_type,
    h.page_start
FROM "2_child_home_iot_controls" h
INNER JOIN "1_parent_iot_common_guidelines" g
    ON h.parent_guideline_id = g.guideline_id

UNION ALL

SELECT
    g.guideline_name,
    'STM32 디버깅 가이드' AS source_type,
    s.topic_name AS item_name,
    s.hardware_or_security AS item_type,
    s.page_start
FROM "3_child_stm32_debug_topics" s
INNER JOIN "1_parent_iot_common_guidelines" g
    ON s.parent_guideline_id = g.guideline_id

UNION ALL

SELECT
    g.guideline_name,
    'ESP32-H2 기술 매뉴얼' AS source_type,
    e.feature_name AS item_name,
    e.hardware_or_security AS item_type,
    e.page_start
FROM "4_child_esp32h2_features" e
INNER JOIN "1_parent_iot_common_guidelines" g
    ON e.parent_guideline_id = g.guideline_id

ORDER BY guideline_name, source_type, page_start;


결과:


[('IoT 장치 특성을 고려한 보안 서비스 경량화', 'ESP32-H2 기술 매뉴얼', 'System and Memory', '하드웨어', 142), ('IoT 장치 특성을 고려한 보안 서비스 경량화', 'ESP32-H2 기술 매뉴얼', 'IO MUX and GPIO Matrix', '하드웨어', 217), ('IoT 장치 특성을 고려한 보안 서비스 경량화', 'ESP32-H2 기술 매뉴얼', 'GPIO Hysteresis', '하드웨어', 227), ('IoT 장치 특성을 고려한 보안 서비스 경량화', 'ESP32-H2 기술 매뉴얼', 'GPIO Power Supply Management', '하드웨어', 228), ('IoT 장치 특성을 고려한 보안 서비스 경량화', 'ESP32-H2 기술 매뉴얼', 'Reset and Clock', '하드웨어', 270), ('IoT 장치 특성을 고려한 보안 서비스 경량화', 'ESP32-H2 기술 매뉴얼', 'Low-Power Management', '하드웨어', 387), ('IoT 장치 특성을 고려한 보안 서비스 경량화', 'ESP32-H2 기술 매뉴얼', 'UART Controller', '하드웨어', 701), ('IoT 장치 특성을 고려한 보안 서비스 경량화', 'ESP32-H2 기술 매뉴얼', 'SPI Controller', '하드웨어', 766), ('IoT 장치 특성을 고려한 보안 서비스 경량화', 'ESP32-H2 기술 매뉴얼', 'I2C Controller', '하드웨어', 838), ('IoT 장치 특성을 고려한 보안 서비스 경량화', 'STM32 디버깅 가이드', '저전력 상태 디버깅', '하드웨어', 50), ('IoT 장치 특성을 고려한 보안 서비스 경량화', 'STM32 디버깅 가이드', 'Microcontroller Clock Output(MCO)', '하드웨어', 87), ('Secure by Default 적용', 'ESP32-H2 기술 매뉴얼', 'Chip Boot Control', 

In [151]:
# TODO: 집계(Aggregation) 쿼리 작성
# 예시: GROUP BY, COUNT 등 사용

aggregation_query = """
SELECT
    g.guideline_name,
    COUNT(DISTINCT h.home_control_id) AS home_iot_count,
    COUNT(DISTINCT s.stm32_topic_id) AS stm32_count,
    COUNT(DISTINCT e.esp32_feature_id) AS esp32_count
FROM "1_parent_iot_common_guidelines" g
LEFT JOIN "2_child_home_iot_controls" h
    ON h.parent_guideline_id = g.guideline_id
LEFT JOIN "3_child_stm32_debug_topics" s
    ON s.parent_guideline_id = g.guideline_id
LEFT JOIN "4_child_esp32h2_features" e
    ON e.parent_guideline_id = g.guideline_id
GROUP BY g.guideline_name
ORDER BY home_iot_count DESC;
"""

print("실행 쿼리:")
print(aggregation_query)
print("\n결과:")

try:
    result = db.run(aggregation_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT
    g.guideline_name,
    COUNT(DISTINCT h.home_control_id) AS home_iot_count,
    COUNT(DISTINCT s.stm32_topic_id) AS stm32_count,
    COUNT(DISTINCT e.esp32_feature_id) AS esp32_count
FROM "1_parent_iot_common_guidelines" g
LEFT JOIN "2_child_home_iot_controls" h
    ON h.parent_guideline_id = g.guideline_id
LEFT JOIN "3_child_stm32_debug_topics" s
    ON s.parent_guideline_id = g.guideline_id
LEFT JOIN "4_child_esp32h2_features" e
    ON e.parent_guideline_id = g.guideline_id
GROUP BY g.guideline_name
ORDER BY home_iot_count DESC;


결과:
[('접근권한 관리·인증·종단간 통신 보안·데이터 암호화', 4, 0, 1), ('다양한 하드웨어 보안기법 적용', 4, 10, 6), ('소프트웨어 취약점 점검 및 보안패치 방안 구현', 2, 1, 0), ('소프트웨어·하드웨어 보안기술 적용 검토', 2, 1, 7), ('시큐어코딩 적용', 1, 0, 0), ('개인정보보호정책 및 보호조치 마련', 1, 0, 0), ('민감정보 보호', 1, 0, 1), ('안전한 보안 프로토콜 및 파라미터 설정', 1, 0, 1), ('취약점 분석 및 보안패치 배포', 1, 0, 0), ('로그기록 저장·관리', 1, 0, 0), ('침입탐지 및 모니터링', 0, 0, 0), ('Secure by Default 적용', 0, 0, 1), ('민감정보 운영정책 투명성 보장', 0, 0, 0), ('보안취약점 및 보호조치 공지', 0, 0,

## 7. Text2SQL 함수 구현

IoT 하드웨어·보안 데이터에 맞는 SQL 생성 프롬프트를 적용합니다.

In [152]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

llm = init_chat_model("gpt-5.4-mini")

def text_to_sql(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문을 SQL로 변환
    """
    system_prompt = f"""
당신은 IoT 디바이스의 하드웨어 설계와 보안 데이터를 다루는 SQL 전문가입니다.
사용자의 자연어 질문을 PostgreSQL SELECT 쿼리로 변환하세요.

데이터베이스 스키마:
{db.table_info}

<데이터베이스 설명>

- "1_parent_iot_common_guidelines":
  IoT 공통 보안 가이드의 상위 기준 테이블입니다.
  guideline_id가 기본 식별자이며,
  principle_name, lifecycle_stage, guideline_name, domain, description 등의 정보를 포함합니다.

- "2_child_home_iot_controls":
  홈·가전 IoT 보안가이드의 세부 보안·하드웨어 점검 항목입니다.
  parent_guideline_id로 "1_parent_iot_common_guidelines".guideline_id를 참조합니다.
  control_name, control_category, applicable_scope,
  hardware_or_security, implementation_summary, page_start 등의 정보를 포함합니다.

- "3_child_stm32_debug_topics":
  STM32 MCU의 하드웨어·디버깅·보안 기능 정보입니다.
  parent_guideline_id로 "1_parent_iot_common_guidelines".guideline_id를 참조합니다.
  topic_name, section_code, topic_type, interface_or_tool,
  hardware_or_security, applicable_scope, page_start 등의 정보를 포함합니다.

- "4_child_esp32h2_features":
  ESP32-H2의 하드웨어 및 보안 기능 정보입니다.
  parent_guideline_id로 "1_parent_iot_common_guidelines".guideline_id를 참조합니다.
  feature_name, feature_category, module_or_interface,
  hardware_or_security, design_use, page_start 등의 정보를 포함합니다.

</데이터베이스 설명>

<테이블 관계>

"1_parent_iot_common_guidelines".guideline_id
    ← "2_child_home_iot_controls".parent_guideline_id

"1_parent_iot_common_guidelines".guideline_id
    ← "3_child_stm32_debug_topics".parent_guideline_id

"1_parent_iot_common_guidelines".guideline_id
    ← "4_child_esp32h2_features".parent_guideline_id

</테이블 관계>

규칙:
- PostgreSQL 문법을 사용하세요.
- SELECT 쿼리만 생성하세요. INSERT, UPDATE, DELETE, DROP, ALTER는 금지합니다.
- 테이블명이 숫자로 시작하므로 테이블명은 반드시 큰따옴표(" ")로 감싸세요.
- 부모와 자식 테이블을 연결할 때 parent_guideline_id = guideline_id 조건으로 JOIN하세요.
- 하드웨어 관련 질문은 hardware_or_security, feature_category, module_or_interface,
  topic_type, interface_or_tool, design_use 등을 우선 활용하세요.
- 보안 관련 질문은 hardware_or_security, control_category, guideline_name,
  domain, description 등을 활용하세요.
- '하드웨어', '보안', '통합'을 구분해야 할 경우 hardware_or_security 컬럼을 사용하세요.
- ESP32-H2 관련 질문은 "4_child_esp32h2_features"를 우선 사용하세요.
- STM32 관련 질문은 "3_child_stm32_debug_topics"를 우선 사용하세요.
- 홈캠, 도어락, 스마트TV 등 홈·가전 IoT 제품 보안 질문은
  "2_child_home_iot_controls"를 우선 사용하세요.
- 공통 보안 원칙이나 상위 가이드 기준이 필요한 경우
  "1_parent_iot_common_guidelines"와 JOIN하세요.
- 여러 문서의 데이터를 함께 비교하는 질문은 UNION ALL 또는 JOIN을 적절히 사용하세요.
- 페이지 관련 질문은 page_start 컬럼을 사용하세요.
- 결과가 너무 많을 가능성이 있으면 필요한 경우 LIMIT을 사용하세요.
- SQL 코드만 반환하세요.
- 설명은 반환하지 마세요.
- 코드 블록(```) 없이 순수 SQL만 반환하세요.
- 반드시 세미콜론(;)으로 끝내세요.

사용 가능한 SQL 문법:
- JOIN (INNER, LEFT, RIGHT, FULL)
- UNION, UNION ALL
- GROUP BY, HAVING
- COUNT, SUM, AVG, MIN, MAX
- 서브쿼리
- WHERE, ORDER BY, LIMIT
- CTE (WITH 절)
- 윈도우 함수
"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]

    response = llm.invoke(messages)
    sql = response.content.strip()

    # 코드 블록 제거
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
        sql = sql.replace("sql", "").replace("```", "").strip()

    return sql

print("✓ Text2SQL 함수 준비 완료")

✓ Text2SQL 함수 준비 완료


## 8. Text2SQL 테스트

IoT 보안 관련 자연어 질문으로 SQL 생성 결과를 테스트합니다.

In [153]:
# IoT 보안 데이터 조회 질문
question = "IoT 기기를 만들 때 부품은 어떻게 연결하고, 해킹을 막으려면 어떤 보안 기능을 같이 써야 하나요?"

print(f"질문: {question}\n")
print("="*80)

# SQL 생성
sql = text_to_sql(question, db)
print(f"\n생성된 SQL:")
print(sql)
print()

# SQL 실행
print("="*80)
print("\n실행 결과:")
try:
    result = db.run(sql)
    print(result)
except Exception as e:
    print(f"실행 오류: {e}")

질문: IoT 기기를 만들 때 부품은 어떻게 연결하고, 해킹을 막으려면 어떤 보안 기능을 같이 써야 하나요?




생성된 SQL:
SELECT
    g.guideline_id,
    g.principle_name,
    g.guideline_code,
    g.guideline_name,
    g.domain,
    g.description,
    g.design_focus,
    g.security_focus,
    g.easy_explanation
FROM "1_parent_iot_common_guidelines" AS g
WHERE g.guideline_name IN (
    'IoT 장치 특성을 고려한 보안 서비스 경량화',
    '접근권한 관리·인증·종단간 통신 보안·데이터 암호화',
    '소프트웨어·하드웨어 보안기술 적용 검토'
)
ORDER BY g.guideline_id;


실행 결과:
[(1, '정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계', 'G01', 'IoT 장치 특성을 고려한 보안 서비스 경량화', '하드웨어_설계', '프로세서 성능, 메모리, 입출력장치, 소비전력 등 장치 자원 수준을 고려하여 필요한 보안 기능을 경량화해 구현', 'MCU 성능, 메모리, 입출력장치, 소비전력 등 기기 자원을 먼저 확인', '기기 성능을 넘지 않는 범위에서 필요한 보안 기능을 적용', '기기의 성능과 전력에 맞춰 하드웨어와 보안 기능의 크기를 정하는 기준'), (2, '정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계', 'G02', '접근권한 관리·인증·종단간 통신 보안·데이터 암호화', '인증_통신_암호화', 'IoT 서비스 환경에 맞는 접근권한, 인증, 통신 보호, 데이터 암호화 방안을 제공', '기기와 외부 장치·서버가 연결되는 통신 경로를 확인', '허가된 사용자·장치만 접근하게 하고 통신과 데이터를 보호', '기기가 다른 장치와 연결될 때 누가 접속할 수 있는지와 데이터 보호를 같이 보는 기준'), (3, '정보보호와 프라이버시 강화를 고려한 IoT 제품·서비스 설계', 'G03', '소프트웨어·하드웨

## 9. IoT 보안 데이터 답변 시스템 (SQL 실행 + 자연어 답변)

조회 결과를 쉬운 IoT 보안 안내로 바꾸는 답변 프롬프트를 적용합니다.

In [154]:
def query_database(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문 → SQL 생성 → 실행 → 자연어 답변
    """
    # 1. SQL 생성
    print(f"[1] SQL 생성 중...")
    sql = text_to_sql(question, db)
    print(f"    {sql}\n")

    # 2. SQL 실행
    print(f"[2] SQL 실행 중...")
    try:
        result = db.run(sql)
        print(f"    실행 완료\n")
    except Exception as e:
        return f"SQL 실행 오류: {e}"

    # 3. 자연어 답변 생성
    print(f"[3] 답변 생성 중...")

    system_prompt = """
당신은 IoT 기기의 하드웨어 설계와 보안을 함께 도와주는 쉬운 설명 전문가입니다.
SQL 조회 결과를 바탕으로 사용자가 실제로 IoT 기기를 만들 때 참고할 수 있도록 답변하세요.

특히 사용자가
"스마트 도어락 만들어줘",
"스마트 홈 기기를 어떻게 만들어요?",
"이런 IoT 기기는 어떻게 제작하나요?"
처럼 제작 방법을 질문하면 단순히 데이터 이름을 나열하지 마세요.

다음 순서로 쉽게 설명하세요.

1. 필요한 하드웨어
   - 어떤 부품이나 기능이 필요한지
   - 각 부품이 무슨 역할을 하는지

2. 연결 및 구성
   - 센서, MCU, 통신 기능 등을 어떤 용도로 연결하는지
   - SQL 결과에서 확인 가능한 범위까지만 설명

3. 보안 설정
   - 외부 연결 포트 보호
   - 디버깅 기능 보호
   - 인증 및 접근 제한
   - 데이터 암호화와 같은 보안 방법
   - 각각 왜 필요한지 쉬운 말로 설명

4. 최종적으로
   - 하드웨어 설계와 보안을 같이 고려했을 때 무엇을 확인해야 하는지 간단히 정리

답변 규칙:
- SQL 결과에 있는 내용을 중심으로 답변하세요.
- SQL 결과에 없는 구체적인 회로 값, 핀 번호, 부품 모델은 만들어내지 마세요.
- 전문용어를 최대한 줄이세요.
- 전문용어가 필요하면 바로 쉬운 뜻을 붙이세요.
  예: JTAG(기기 내부를 점검하는 연결 통로)
- 기능 이름만 나열하지 말고 "어디에 쓰는지"를 설명하세요.
- 보안 기능은 "무엇을 막기 위한 것인지" 설명하세요.
- 하드웨어와 보안을 따로 떼어 설명하지 말고 서로 연결해서 설명하세요.
- 초보자가 읽어도 이해할 수 있는 쉬운 한국어를 사용하세요.
- 한 문장을 짧게 작성하세요.
- SQL이나 데이터베이스라는 표현은 최종 답변에서 굳이 언급하지 마세요.
- 정보가 부족한 부분은 "현재 자료에서는 구체적인 방법까지 확인하기 어렵습니다."라고 안내하세요.
"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
        질문: {question}

        실행한 SQL:
        {sql}

        쿼리 결과:
        {result}

        위 결과를 바탕으로 질문에 답변해주세요.
        """)
    ]

    response = llm.invoke(messages)
    return response.content

print("✓ 완전한 Text2SQL 시스템 준비 완료")

✓ 완전한 Text2SQL 시스템 준비 완료


In [155]:
from IPython.display import Markdown, display

# IoT 하드웨어·보안 통합 질문
question = "스마트 도어락 제작 방법 알려줘"

print(f"질문: {question}\n")
print("="*80 + "\n")

answer = query_database(question, db)

print("\n" + "="*80)
print("\n답변:")
display(Markdown(answer))

질문: 스마트 도어락 제작 방법 알려줘


[1] SQL 생성 중...
    SELECT
  p.guideline_id,
  p.guideline_name,
  p.principle_name,
  p.lifecycle_stage,
  p.domain,
  p.description,
  c.control_name,
  c.control_category,
  c.applicable_scope,
  c.hardware_or_security,
  c.implementation_summary,
  c.hardware_design_point,
  c.security_purpose,
  c.easy_explanation,
  c.page_start
FROM "2_child_home_iot_controls" c
JOIN "1_parent_iot_common_guidelines" p
  ON c.parent_guideline_id = p.guideline_id
WHERE c.applicable_scope ILIKE '%도어락%'
   OR c.control_name ILIKE '%도어락%'
   OR c.description ILIKE '%도어락%'
   OR c.easy_explanation ILIKE '%도어락%'
ORDER BY c.page_start ASC;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...


답변:


스마트 도어락을 만들 때는, **문을 열고 닫는 기능**만 보는 것이 아니라, **외부에서 뜯거나 조작하려는 시도까지 막는 구조**를 함께 생각해야 합니다.

## 1. 필요한 하드웨어

현재 자료에서 확인되는 핵심은 **외부 조작 확인 및 분해 방지 메커니즘**입니다.

즉, 스마트 도어락에는 다음 같은 하드웨어가 필요합니다.

- **도어락 본체 케이스**
  - 기기 내부 부품을 넣는 몸체입니다.
  - 외부에서 쉽게 열리지 않게 만들어야 합니다.

- **분해 방지 구조**
  - 누군가 억지로 열거나 뜯을 때 흔적이 남도록 하는 구조입니다.
  - 이는 내부 부품과 저장된 정보가 밖으로 노출되는 것을 줄여줍니다.

- **외부 조작 확인 기능**
  - 누군가 임의로 만지거나 건드렸는지 확인하는 역할입니다.
  - 도어락처럼 손이 쉽게 닿는 제품에 중요합니다.

이 자료에서는 센서, 통신 모듈, MCU(기기를 제어하는 작은 컴퓨터) 같은 구체 부품은 확인되지 않습니다.  
그래서 **현재 자료에서는 구체적인 방법까지 확인하기 어렵습니다.**

## 2. 연결 및 구성

이번 결과에서 확인되는 연결 관점은, **도어락의 내부 기능이 외부 물리 조작에 노출되지 않게 구성하는 것**입니다.

쉽게 말하면:

- **케이스와 내부 보드**를 분리해서 생각합니다.
- 내부 부품이 쉽게 보이거나 손대기 어렵게 만듭니다.
- 외부에서 분해 시도가 있으면 이를 감지하거나 흔적이 남게 합니다.

이런 구성은 도어락이 단순히 작동하는 것보다 더 중요합니다.  
왜냐하면 도어락은 **문을 여는 장치**이면서 동시에 **침입 대상**이 될 수 있기 때문입니다.

다만, 이 결과에는 센서가 어떻게 연결되는지, 통신 기능을 어떻게 구성하는지는 나오지 않습니다.  
그래서 그 부분은 **현재 자료에서는 구체적인 방법까지 확인하기 어렵습니다.**

## 3. 보안 설정

이번 자료에서 가장 중요한 보안은 **Tamper Proofing(훼손 방지)** 입니다.  
이것은 **기기를 뜯거나 조작했을 때 쉽게 원래 상태로 돌릴 수 없게 하거나, 흔적이 남게 하는 방식**입니다.

### 왜 필요한가요?

- **비인가 분해 방지**
  - 도어락 내부를 열어보면 인증 정보나 제어부를 건드릴 수 있습니다.
  - 이를 막아야 도어락이 뚫리는 위험이 줄어듭니다.

- **외부 조작 방지**
  - 버튼, 덮개, 연결부를 억지로 만지는 행위를 줄입니다.
  - 기기가 잘못 동작하는 것을 막습니다.

- **중요 데이터와 기능 보호**
  - 내부 정보가 밖으로 노출되면 도어락 전체가 위험해질 수 있습니다.
  - 그래서 물리적으로도 보호가 필요합니다.

또한 자료에는 **펌웨어·코드 암호화, 실행코드 영역 제어, 역공학 방지** 같은 하드웨어 보안기법이 함께 언급되어 있습니다.  
하지만 이번 결과에서는 도어락에 어떤 방식으로 적용하는지까지는 나오지 않습니다.  
즉, **구체적인 구현 방법은 현재 자료에서는 확인하기 어렵습니다.**

## 4. 최종 정리

스마트 도어락을 설계할 때는 다음을 함께 확인해야 합니다.

- **케이스가 쉽게 분해되지 않는가**
- **외부에서 조작했을 때 흔적이 남는가**
- **내부 중요 정보가 물리적으로 노출되지 않는가**
- **뜯기거나 만져졌을 때 기능과 데이터를 보호할 수 있는가**

한마디로 정리하면,  
스마트 도어락은 **열쇠 기능만 만드는 것이 아니라, 뜯기 어려운 구조와 조작 방지 설계까지 같이 해야 안전합니다.**

원하시면 다음 단계로  
**“스마트 도어락 설계 체크리스트”** 형태로 더 쉽게 정리해드릴게요.

## 10. 다양한 질문으로 테스트

**TODO: 최소 5개 이상의 다양한 질문으로 테스트하세요**

In [156]:
# 기능별 IoT 하드웨어·보안 테스트 질문
# 기본 조회, JOIN, 집계, 정렬 등 다양한 유형의 질문 포함

questions = [
    "스마트 도어락을 만들 때 어떤 하드웨어와 보안 기능이 필요한가요?",
    "ESP32-H2에서 부품 연결에 쓰는 기능과 해킹을 막는 보안 기능을 같이 알려주세요.",
    "STM32에서 기기 내부를 보호하기 위해 사용할 수 있는 기능은 무엇인가요?",
    "IoT 공통 보안 가이드별로 연결된 홈가전, STM32, ESP32-H2 항목이 각각 몇 개인가요?",
    "하드웨어와 관련된 기능들을 페이지 순서대로 보여주세요."
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print("="*80 + "\n")

    try:
        answer = query_database(q, db)
        print("\n답변:")
        display(Markdown(answer))
    except Exception as e:
        print(f"오류: {e}")


질문: 스마트 도어락을 만들 때 어떤 하드웨어와 보안 기능이 필요한가요?

[1] SQL 생성 중...
    SELECT
  c.control_name,
  c.control_category,
  c.applicable_scope,
  c.hardware_or_security,
  c.implementation_summary,
  c.hardware_design_point,
  c.security_purpose,
  c.easy_explanation,
  p.principle_name,
  p.lifecycle_stage,
  p.guideline_name,
  p.domain
FROM "2_child_home_iot_controls" c
LEFT JOIN "1_parent_iot_common_guidelines" p
  ON c.parent_guideline_id = p.guideline_id
WHERE c.applicable_scope ILIKE '%도어락%'
   OR c.applicable_scope ILIKE '%홈·가전 IoT 제품%'
   OR c.description ILIKE '%도어락%'
   OR c.easy_explanation ILIKE '%도어락%'
ORDER BY c.page_start ASC;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


스마트 도어락은 **문을 여닫는 기능**과 **외부 조작을 막는 보안**을 함께 생각해야 합니다.  
현재 자료에서는 구체적인 센서 이름이나 회로까지는 확인하기 어렵습니다.  
대신, 어떤 하드웨어와 보안 기능을 준비해야 하는지 쉽게 정리해드리겠습니다.

## 1. 필요한 하드웨어

### 1) 도어락 본체를 움직이는 부분
- 문을 잠그고 푸는 **구동부**가 필요합니다.
- 이 부분이 실제로 걸쇠를 움직여 문을 열고 닫습니다.
- 스마트 도어락의 핵심 역할입니다.

### 2) 제어를 담당하는 MCU
- MCU는 **기기를 두뇌처럼 제어하는 작은 컴퓨터**입니다.
- 비밀번호, 인증 결과, 잠금 상태를 처리합니다.
- 센서와 통신 기능도 함께 관리합니다.

### 3) 통신 기능
- 스마트폰이나 서버와 연결하려면 통신 기능이 필요합니다.
- 원격 제어, 상태 확인, 알림 전송에 쓰입니다.
- 다만 어떤 통신 방식을 쓰는지는 현재 자료에서는 구체적으로 확인하기 어렵습니다.

### 4) 외부 조작을 감지하거나 막는 구조
- 도어락은 사람이 손으로 직접 만질 수 있습니다.
- 그래서 **케이스 분해 여부나 외부 조작 흔적을 확인할 수 있는 구조**가 필요합니다.
- 이것은 단순한 편의 기능이 아니라 보안 기능과 바로 연결됩니다.

---

## 2. 연결 및 구성

### 1) MCU와 잠금 구동부 연결
- MCU가 인증 결과를 판단하면 잠금 장치를 움직이게 합니다.
- 즉, “맞는 사용자인가?”를 확인한 뒤 문을 열도록 연결합니다.
- 이 구조가 있어야 인증과 실제 동작이 이어집니다.

### 2) 입력부와 MCU 연결
- 비밀번호 입력, 카드 인식, 앱 요청 같은 입력이 있으면 MCU가 받아야 합니다.
- 현재 자료에는 입력 방식의 종류가 나오지 않아 구체적인 연결 방식까지는 확인하기 어렵습니다.
- 하지만 핵심은 **입력값을 MCU가 안전하게 처리하는 것**입니다.

### 3) 통신부와 MCU 연결
- 외부 기기와 통신하는 부분도 MCU가 제어합니다.
- 원격 제어 기능이 있더라도, 무조건 열리게 두면 안 됩니다.
- 그래서 통신부는 인증된 명령만 전달되도록 구성해야 합니다.

### 4) 외부 조작 감지 구조
- 케이스가 열리거나 뜯기면 이를 확인할 수 있어야 합니다.
- 이 정보는 MCU가 받아 경고를 띄우거나 기능을 제한하는 데 쓸 수 있습니다.
- 도어락처럼 물리적으로 접근 가능한 제품에는 특히 중요합니다.

---

## 3. 보안 설정

### 1) 시큐어코딩 적용
- 프로그램을 안전한 규칙으로 작성하는 것입니다.
- 입력값 처리 실수를 줄여서 해킹이나 오류를 막습니다.
- 도어락은 인증 정보를 다루므로 작은 실수도 위험할 수 있습니다.

### 2) 알려진 보안취약점 점검 및 제거
- 이미 공개된 약점을 제품에 남기지 않는 것입니다.
- 기기에 들어간 OS, 라이브러리, 통신 모듈이 오래되면 위험해집니다.
- 그래서 사용한 구성요소를 관리하고, 문제점이 있으면 제거해야 합니다.

### 3) 최신 3rd party 소프트웨어 사용
- 외부에서 가져온 프로그램도 최신 보안패치가 들어간 버전을 써야 합니다.
- 오래된 프로그램은 공격에 약할 수 있습니다.
- 도어락은 집 안 출입과 연결되므로, 이런 약점이 있으면 바로 위험해집니다.

### 4) 외부 조작 확인 및 분해 방지
- 케이스를 뜯거나 내부를 건드리는 행동을 막거나 확인해야 합니다.
- 비인가 분해를 하면 중요한 데이터나 기능이 노출될 수 있습니다.
- 도어락은 집 문에 붙어 있으므로, 물리적 공격도 꼭 대비해야 합니다.

### 5) 인증과 접근 제한
- 문을 여는 기능은 아무나 쓰면 안 됩니다.
- 사용자 인증이 끝난 뒤에만 동작하도록 제한해야 합니다.
- 원격 기능도 관리자만 바꾸게 하는 식의 제한이 필요합니다.
- 이렇게 해야 오작동이나 무단 개방을 막을 수 있습니다.

### 6) 데이터 보호
- 인증 정보나 상태 정보는 외부에 쉽게 보이면 안 됩니다.
- 통신 중인 정보도 안전하게 다뤄야 합니다.
- 현재 자료에는 암호화 방식의 구체적인 내용은 없지만, 이런 보호가 필요하다는 점은 분명합니다.

---

## 4. 최종 정리

스마트 도어락을 만들 때는 다음을 같이 봐야 합니다.

- **구동부**가 문을 실제로 열고 닫는지.
- **MCU**가 인증과 제어를 안전하게 처리하는지.
- **통신 기능**이 있어도 인증 없이 동작하지 않는지.
- **분해 방지 구조**로 물리적 조작을 막을 수 있는지.
- **시큐어코딩**, **취약점 점검**, **최신 소프트웨어 사용**으로 소프트웨어 약점을 줄였는지.
- **인증과 접근 제한**이 제대로 들어가 있는지.

한마디로,  
**스마트 도어락은 “잘 열리는 것”보다 “허가된 사람만 안전하게 열 수 있는 것”이 더 중요합니다.**

원하시면 다음 단계로  
**스마트 도어락 구조를 블록도처럼 쉽게 그려서** 설명해드릴게요.


질문: ESP32-H2에서 부품 연결에 쓰는 기능과 해킹을 막는 보안 기능을 같이 알려주세요.

[1] SQL 생성 중...
    SELECT
    e.esp32_feature_id,
    e.feature_name,
    e.feature_category,
    e.module_or_interface,
    e.hardware_or_security,
    e.design_use,
    e.description,
    e.hardware_design_point,
    e.security_purpose,
    e.easy_explanation,
    p.guideline_id,
    p.guideline_name,
    p.domain,
    p.description AS guideline_description
FROM "4_child_esp32h2_features" e
LEFT JOIN "1_parent_iot_common_guidelines" p
    ON e.parent_guideline_id = p.guideline_id
WHERE e.hardware_or_security IN ('하드웨어', '통합', '보안')
ORDER BY CASE e.hardware_or_security
    WHEN '하드웨어' THEN 1
    WHEN '통합' THEN 2
    WHEN '보안' THEN 3
    ELSE 4
END, e.page_start;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


ESP32-H2에서 **부품을 연결하는 기능**과 **해킹을 막는 보안 기능**을 같이 보면,  
핵심은 **“어떤 부품을 어디에 붙일지 정하고, 그 연결 경로를 안전하게 막는 것”**입니다.

---

## 1. 필요한 하드웨어

ESP32-H2에서 부품 연결에 많이 쓰는 기능은 다음과 같습니다.

- **GPIO / IO MUX / GPIO Matrix**
  - 센서나 버튼, LED 같은 부품을 어떤 핀에 연결할지 정하는 기능입니다.
  - 그냥 핀에 꽂는 것만이 아니라, 내부에서 신호가 어느 핀으로 갈지 조정합니다.
  - 그래서 여러 주변장치를 유연하게 붙일 수 있습니다.

- **UART**
  - 두 장치가 간단하게 데이터를 주고받는 통신 기능입니다.
  - 보통 디버그 로그를 보거나, 외부 장치와 간단히 연결할 때 씁니다.
  - 개발할 때는 편하지만, 완제품에서는 노출되면 위험할 수 있습니다.

- **SPI**
  - 센서, 화면 컨트롤러, 외부 Flash, RAM처럼 빠르게 통신해야 하는 부품을 연결할 때 씁니다.
  - 외부 메모리나 빠른 주변장치를 붙이는 용도에 맞습니다.

- **I2C**
  - 두 개의 신호선으로 여러 센서나 주변장치를 함께 연결할 수 있는 기능입니다.
  - 센서, 보안칩 같은 부품을 붙일 때 자주 씁니다.

- **External Flash / 내부 메모리 구조**
  - 프로그램과 데이터를 어디에 저장할지 정하는 데 필요합니다.
  - 외부 Flash는 용량을 늘리는 데 좋지만, 중요한 데이터가 그대로 보이지 않게 보호가 필요합니다.

- **GPIO 히스테리시스**
  - 버튼이나 센서 입력이 잡음 때문에 흔들리지 않게 해주는 기능입니다.
  - 입력이 자꾸 바뀌면 기기가 잘못 동작할 수 있어서 중요합니다.

- **전원 관리 / 저전력 기능**
  - 배터리를 쓰는 기기라면 꼭 필요합니다.
  - 일부 핀은 절전 중에도 깨우는 용도로 쓸 수 있습니다.
  - 그래서 “항상 켜둘 것”과 “필요할 때만 켤 것”을 나눠 설계할 수 있습니다.

- **리셋·클록**
  - MCU가 어떤 속도로 동작하고 언제 다시 시작할지 정하는 기능입니다.
  - 이상이 생겼을 때 안전하게 다시 시작하도록 돕습니다.

---

## 2. 연결 및 구성

부품 연결은 보통 이렇게 생각하면 쉽습니다.

- **센서나 버튼**
  - GPIO에 연결합니다.
  - 입력이 흔들릴 수 있으니 GPIO 히스테리시스를 함께 쓰면 좋습니다.

- **통신이 필요한 부품**
  - 간단한 통신은 UART를 씁니다.
  - 센서나 외부 메모리는 I2C나 SPI로 연결합니다.
  - I2C는 선이 적고 여러 부품을 같이 붙이기 좋습니다.
  - SPI는 더 빠른 통신이 필요할 때 유리합니다.

- **외부 메모리**
  - SPI 기반 외부 Flash와 연결할 수 있습니다.
  - 프로그램 코드나 데이터 저장에 쓰입니다.
  - 다만 외부 메모리에 민감한 정보가 있으면 보호가 필요합니다.

- **디버그와 프로그래밍**
  - USB Serial/JTAG나 UART가 개발할 때 유용합니다.
  - 하지만 제품이 완성된 뒤에는 그대로 열어두지 않는 것이 중요합니다.

- **저전력 구성**
  - PMU와 전원 도메인을 이용해 일부 기능만 살리고 나머지는 재울 수 있습니다.
  - 배터리 제품에서는 이 부분이 매우 중요합니다.

---

## 3. 보안 설정

부품 연결이 편해질수록, 같이 열리는 통로도 늘어납니다.  
그래서 연결 기능과 보안을 같이 봐야 합니다.

- **JTAG 경로 제어**
  - JTAG는 기기 내부를 점검하는 연결 통로입니다.
  - 개발 때는 유용하지만, 완제품에서는 공격자가 내부를 들여다보는 통로가 될 수 있습니다.
  - 그래서 JTAG 신호 소스를 제어하거나 비활성화해야 합니다.

- **USB Serial/JTAG 제한**
  - USB 한 연결로 프로그래밍과 디버깅을 할 수 있습니다.
  - 편리하지만, 출시 후에는 이 경로가 공격 통로가 될 수 있습니다.
  - 필요하지 않다면 제한하거나 꺼야 합니다.

- **UART 디버그 경로 관리**
  - UART는 로그 출력용으로 자주 쓰입니다.
  - 하지만 로그에 내부 정보가 많이 나오면 위험합니다.
  - 제품에서는 불필요한 출력이 남지 않게 관리해야 합니다.

- **부트 제어**
  - 기기가 켜질 때 다운로드 모드로 쉽게 들어가면 위험할 수 있습니다.
  - eFuse와 GPIO 설정으로 부팅 경로를 관리해 원하지 않는 시작 방식을 막습니다.

- **eFuse**
  - 한번 정한 중요한 보안 설정을 칩 안에 저장하는 영역입니다.
  - JTAG 차단, 부트 설정, 키 관련 설정 같은 걸 관리할 때 씁니다.
  - 잘못 설정하면 되돌리기 어렵기 때문에 신중해야 합니다.

- **외부 메모리 암호화**
  - 외부 Flash에 있는 코드와 민감 데이터를 XTS-AES 방식으로 암호화합니다.
  - 외부 메모리를 떼어내도 내용이 바로 보이지 않게 하는 목적입니다.

- **암호 하드웨어**
  - AES, ECC, RSA, SHA, HMAC, DSA, ECDSA 같은 기능이 있습니다.
  - 중요한 이유는 암호 계산을 더 빠르고 안정적으로 처리할 수 있기 때문입니다.
  - 예를 들어 AES는 데이터 암호화에 쓰고, SHA는 데이터가 바뀌지 않았는지 확인할 때 씁니다.
  - ECDSA나 RSA는 신뢰할 수 있는 장치인지 확인하는 데 도움이 됩니다.
  - HMAC은 메시지가 중간에 바뀌지 않았는지 보는 데 유용합니다.

- **Random Number Generator**
  - 예측하기 어려운 숫자를 만들어 키나 인증값 생성에 씁니다.
  - 보안에서는 랜덤값이 약하면 전체가 약해질 수 있어서 중요합니다.

- **전원 이상 감지**
  - 전압 글리치나 브라운아웃을 감지하면 리셋할 수 있습니다.
  - 공격자가 전원을 일부러 흔들어 보안 처리를 깨는 상황을 막는 데 도움이 됩니다.

- **접근권한 제어**
  - 메모리와 주변장치에 누가 읽고 쓰고 실행할 수 있는지 나눠 관리합니다.
  - 기기 내부에서도 모든 기능이 모든 곳에 접근하면 안 됩니다.
  - 그래서 중요한 영역을 분리하는 게 좋습니다.

---

## 4. 최종 정리

ESP32-H2로 IoT 기기를 만들 때는 이렇게 보면 됩니다.

- **부품 연결**
  - GPIO/IO MUX/GPIO Matrix로 핀 역할을 정합니다.
  - UART, SPI, I2C로 주변장치를 붙입니다.
  - GPIO 히스테리시스로 입력 흔들림을 줄입니다.
  - 저전력 기능으로 배터리 소모를 줄입니다.

- **보안**
  - JTAG, USB Serial/JTAG, UART 디버그 통로를 출시 전에 꼭 점검합니다.
  - eFuse와 부트 제어로 원하지 않는 접근을 막습니다.
  - 외부 Flash는 암호화해서 저장합니다.
  - 암호 하드웨어와 난수 생성기를 활용해 인증과 데이터 보호를 강화합니다.
  - 전원 이상 감지와 접근권한 제어도 함께 넣어야 합니다.

즉, **“어떤 부품을 연결할지”와 “그 연결이 밖으로 새지 않게 할지”를 같이 설계해야** 합니다.  
현재 자료에서는 구체적인 회로 값이나 핀 번호까지는 확인하기 어렵습니다.

원하시면 다음 단계로  
**“ESP32-H2 기반 스마트 도어락 예시”**처럼 실제 제품 형태로 쉽게 묶어서 설명해드릴 수 있습니다.


질문: STM32에서 기기 내부를 보호하기 위해 사용할 수 있는 기능은 무엇인가요?

[1] SQL 생성 중...
    SELECT
  p.guideline_id,
  p.guideline_name,
  p.domain,
  t.topic_name,
  t.section_code,
  t.topic_type,
  t.interface_or_tool,
  t.hardware_or_security,
  t.description,
  t.security_purpose,
  t.easy_explanation
FROM "3_child_stm32_debug_topics" AS t
JOIN "1_parent_iot_common_guidelines" AS p
  ON t.parent_guideline_id = p.guideline_id
WHERE t.hardware_or_security = '하드웨어'
  AND (
    t.topic_name ILIKE '%보안%'
    OR t.description ILIKE '%보안%'
    OR t.security_purpose ILIKE '%보안%'
    OR t.easy_explanation ILIKE '%보안%'
  )
ORDER BY t.page_start ASC;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


STM32에서 기기 내부를 보호할 때, 현재 확인된 자료에서는 **STM32CubeMX**를 활용해 **핀과 주변장치 설정을 설계 단계에서 같이 점검하는 방법**이 확인됩니다.

1. 필요한 하드웨어  
- 여기서 핵심은 **MCU 주변장치와 핀 구성**입니다.  
- STM32CubeMX는 어떤 핀을 어떤 기능에 쓸지 정하는 데 도움을 줍니다.  
- 즉, 보안 기능을 넣을 때도 **어떤 핀을 외부에 열어둘지**, **어떤 주변장치를 사용할지** 먼저 정리하는 데 쓰입니다.  
- 예를 들어, 디버깅이나 통신처럼 내부에 접근할 수 있는 기능이 있으면 설계 단계에서 함께 확인해야 합니다.

2. 연결 및 구성  
- 현재 자료에서는 **핀·주변장치 설정**을 통해 초기화 코드를 만드는 역할까지만 확인됩니다.  
- 따라서 센서나 통신 기능을 어떻게 연결하는지의 구체적인 방법은 확인하기 어렵습니다.  
- 다만 보안 관점에서는, 내부 보호를 위해 **필요한 기능만 핀에 연결하고**, 쓰지 않는 연결은 열어두지 않는 식으로 설계하는 것이 중요합니다.  
- STM32CubeMX는 이런 구성을 미리 정리하는 데 도움이 됩니다.

3. 보안 설정  
- 현재 결과에서 확인되는 보안 관련 핵심은 **보안 기능을 사용할 핀과 주변장치 설정을 설계 단계에서 함께 확인한다**는 점입니다.  
- 왜 필요하냐면, 내부를 보호하려면 외부에서 들어올 수 있는 길을 줄여야 하기 때문입니다.  
- 핀 설정을 잘못하면 원치 않는 기능이 열려서 내부 상태를 볼 수 있습니다.  
- 그래서 **외부 연결 포트 보호**, **디버깅 기능 보호**, **접근 제한** 같은 내용을 설계 단계에서 함께 점검해야 합니다.  
- 다만 현재 자료에서는 이러한 보안 기능을 STM32에서 어떻게 켜는지의 구체적인 방법까지는 확인하기 어렵습니다.

4. 최종적으로  
- 지금 자료로 확인되는 핵심은 **STM32CubeMX로 핀과 주변장치 구성을 먼저 정리하고, 보안에 필요한 설정이 함께 열려 있는지 확인하는 것**입니다.  
- 즉, 하드웨어 설계와 보안을 따로 보지 말고, **어떤 핀과 기능을 열 것인지**를 같이 검토해야 합니다.  
- 현재 자료에서는 구체적인 차단 방법이나 세부 설정은 부족하므로, 자세한 적용 방법까지는 확인하기 어렵습니다.


질문: IoT 공통 보안 가이드별로 연결된 홈가전, STM32, ESP32-H2 항목이 각각 몇 개인가요?

[1] SQL 생성 중...
    SELECT
    p.guideline_id,
    p.guideline_name,
    COUNT(DISTINCT h.home_control_id) AS home_gadget_count,
    COUNT(DISTINCT s.stm32_topic_id) AS stm32_count,
    COUNT(DISTINCT e.esp32_feature_id) AS esp32h2_count
FROM "1_parent_iot_common_guidelines" p
LEFT JOIN "2_child_home_iot_controls" h
    ON h.parent_guideline_id = p.guideline_id
LEFT JOIN "3_child_stm32_debug_topics" s
    ON s.parent_guideline_id = p.guideline_id
LEFT JOIN "4_child_esp32h2_features" e
    ON e.parent_guideline_id = p.guideline_id
GROUP BY p.guideline_id, p.guideline_name
ORDER BY p.guideline_id;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


아래처럼 각 보안 가이드별로 연결된 항목 수를 볼 수 있습니다.  
홈가전, STM32, ESP32-H2가 각각 몇 개인지도 함께 정리했습니다.

1. **IoT 장치 특성을 고려한 보안 서비스 경량화**
   - 홈가전: **0개**
   - STM32: **2개**
   - ESP32-H2: **9개**

2. **접근권한 관리·인증·종단간 통신 보안·데이터 암호화**
   - 홈가전: **4개**
   - STM32: **0개**
   - ESP32-H2: **1개**

3. **소프트웨어·하드웨어 보안기술 적용 검토**
   - 홈가전: **2개**
   - STM32: **1개**
   - ESP32-H2: **7개**

4. **민감정보 보호**
   - 홈가전: **1개**
   - STM32: **0개**
   - ESP32-H2: **1개**

5. **민감정보 운영정책 투명성 보장**
   - 홈가전: **0개**
   - STM32: **0개**
   - ESP32-H2: **0개**

6. **시큐어코딩 적용**
   - 홈가전: **1개**
   - STM32: **0개**
   - ESP32-H2: **0개**

7. **소프트웨어 취약점 점검 및 보안패치 방안 구현**
   - 홈가전: **2개**
   - STM32: **1개**
   - ESP32-H2: **0개**

8. **다양한 하드웨어 보안기법 적용**
   - 홈가전: **4개**
   - STM32: **10개**
   - ESP32-H2: **6개**

9. **Secure by Default 적용**
   - 홈가전: **0개**
   - STM32: **0개**
   - ESP32-H2: **1개**

10. **안전한 보안 프로토콜 및 파라미터 설정**
    - 홈가전: **1개**
    - STM32: **0개**
    - ESP32-H2: **1개**

11. **취약점 분석 및 보안패치 배포**
    - 홈가전: **1개**
    - STM32: **0개**
    - ESP32-H2: **0개**

12. **보안취약점 및 보호조치 공지**
    - 홈가전: **0개**
    - STM32: **0개**
    - ESP32-H2: **0개**

13. **개인정보보호정책 및 보호조치 마련**
    - 홈가전: **1개**
    - STM32: **0개**
    - ESP32-H2: **0개**

14. **침입탐지 및 모니터링**
    - 홈가전: **0개**
    - STM32: **0개**
    - ESP32-H2: **0개**

15. **로그기록 저장·관리**
    - 홈가전: **1개**
    - STM32: **0개**
    - ESP32-H2: **0개**

원하시면 다음 단계로  
**“어떤 보안 가이드가 어떤 기기군에 가장 많이 연결되어 있는지”**  
또는 **“홈가전/STM32/ESP32-H2별로 보안 포인트를 쉬운 말로 설명”** 해드릴 수 있습니다.


질문: 하드웨어와 관련된 기능들을 페이지 순서대로 보여주세요.

[1] SQL 생성 중...
    SELECT
    'home_iot_controls' AS source_table,
    c.page_start,
    c.control_name AS item_name,
    c.hardware_or_security,
    c.description,
    p.guideline_name,
    p.domain
FROM "2_child_home_iot_controls" c
LEFT JOIN "1_parent_iot_common_guidelines" p
    ON c.parent_guideline_id = p.guideline_id
WHERE c.hardware_or_security = '하드웨어'

UNION ALL

SELECT
    'stm32_debug_topics' AS source_table,
    s.page_start,
    s.topic_name AS item_name,
    s.hardware_or_security,
    s.description,
    p.guideline_name,
    p.domain
FROM "3_child_stm32_debug_topics" s
LEFT JOIN "1_parent_iot_common_guidelines" p
    ON s.parent_guideline_id = p.guideline_id
WHERE s.hardware_or_security = '하드웨어'

UNION ALL

SELECT
    'esp32h2_features' AS source_table,
    e.page_start,
    e.feature_name AS item_name,
    e.hardware_or_security,
    e.description,
    p.guideline_name,
    p.domain
FROM "4_child_esp32h2_features" e
LEFT JOIN "1_pa

페이지 순서대로, 하드웨어 관련 기능만 정리하면 아래와 같습니다.

1. ST-LINK 디버그 프로브  
   - STM32 MCU와 JTAG(기기 내부를 점검하는 연결 통로) 또는 SWD(간단한 디버그 연결 방식)로 통신하는 장치입니다.  
   - 보드의 상태를 점검하고, 프로그램을 넣고, 문제가 생겼을 때 원인을 찾는 데 씁니다.  
   - 보안 측면에서는 외부 디버그 연결이 열려 있으면 내부 정보가 보일 수 있으니, 개발이 끝나면 접근을 막는 것이 중요합니다.  

2. STM32CubeMX  
   - MCU 주변장치와 핀 구성을 정하고, 초기화 코드를 만들어 주는 도구입니다.  
   - 어떤 핀을 센서, 통신, 출력에 쓸지 정할 때 필요합니다.  
   - 보안 측면에서는 핀 설정이 잘못되면 디버그나 외부 연결이 예상치 않게 열릴 수 있어, 구성 내용을 함께 확인해야 합니다.  

3. SWD/JTAG 핀 연결  
   - 보드와 디버거를 연결하는 핀 구성입니다.  
   - 개발할 때는 기기 내부를 확인하는 데 쓰고, 양산 후에는 불필요하게 노출되지 않게 해야 합니다.  
   - 물리적으로 쉽게 닿는 위치에 두면 외부 침입에 약해질 수 있습니다.  

4. Reset 및 Connection Mode  
   - 소프트웨어 리셋, 하드웨어 리셋, 코어 리셋과 연결 모드를 다룹니다.  
   - 기기가 멈췄을 때 다시 시작시키거나, 디버깅이 잘 되도록 연결 상태를 바꾸는 데 필요합니다.  
   - 보안 측면에서는 리셋과 연결 방식이 디버깅 우회 통로가 되지 않도록 제한이 필요합니다.  

5. 저전력 상태 디버깅  
   - 기기가 절전 상태일 때도 디버깅할 수 있게 연결과 동작을 다룹니다.  
   - 배터리로 오래 쓰는 IoT 기기에서 상태 확인에 중요합니다.  
   - 다만 절전 중 내부 상태가 더 오래 유지될 수 있으니, 외부 접근을 막는 설정이 필요합니다.  

6. Printf via UART  
   - UART(기기끼리 글자처럼 데이터를 주고받는 방식)나 Virtual COM Port로 실행 로그를 출력합니다.  
   - 동작 중 어떤 일이 일어나는지 확인할 때 씁니다.  
   - 보안 측면에서는 로그에 비밀번호나 내부 정보가 남지 않도록 주의해야 합니다.  

7. Printf via SWO/SWV  
   - SWD 기반 Serial Wire Viewer와 SWO를 사용해 실행 정보를 확인합니다.  
   - UART보다 디버그용으로 더 직접적인 확인에 쓰입니다.  
   - 개발용 기능이므로, 배포 후에는 외부에서 쉽게 읽지 못하게 해야 합니다.  

8. Microcontroller Clock Output(MCO)  
   - MCU의 클록(기기의 동작 기준 신호)을 외부 핀으로 내보내는 기능입니다.  
   - 클록이 제대로 나오는지 확인할 때 도움됩니다.  
   - 외부로 신호를 내보내는 만큼, 불필요하면 끄는 것이 좋습니다.  

9. 하드웨어 핀 탐색  
   - 보드에서 핀 상태와 연결을 확인하며 디버깅하는 방법입니다.  
   - 어떤 핀이 어떤 역할인지 찾을 때 유용합니다.  
   - 물리 접근이 가능하면 내부 연결을 파악당할 수 있어, 보드 보호가 중요합니다.  

10. System and Memory  
   - 내부 ROM·SRAM 구조와 외부 Flash 연결, 캐시 구조를 설명합니다.  
   - 프로그램과 데이터를 어디에 둘지 정할 때 필요합니다.  
   - 보안 측면에서는 외부 메모리 연결이 있으면 읽기와 변조 가능성도 함께 고려해야 합니다.  

11. IO MUX and GPIO Matrix  
   - GPIO 입력·출력과 주변장치 신호를 어떤 핀으로 보낼지 정하는 기능입니다.  
   - 센서, 버튼, 통신선을 원하는 핀에 배치할 때 사용합니다.  
   - 잘못 설정하면 디버그용 신호가 일반 핀으로 노출될 수 있어 주의해야 합니다.  

12. GPIO Hysteresis  
   - GPIO 입력에서 노이즈(잡음) 때문에 값이 흔들리는 것을 줄여 줍니다.  
   - 버튼이나 센서처럼 신호가 불안정할 때 도움이 됩니다.  
   - 보안보다는 안정성에 가깝지만, 오동작을 줄여 예기치 않은 동작을 막는 데도 도움됩니다.  

13. GPIO Power Supply Management  
   - GPIO별 전원 도메인과 Light-sleep, Deep-sleep에서 깨우는 핀을 설명합니다.  
   - 배터리 절약이 중요한 기기에서 웨이크업 신호를 맡깁니다.  
   - 외부 신호로 쉽게 깨워질 수 있으니, 어떤 핀을 깨우기용으로 쓸지 신중히 정해야 합니다.  

14. Reset and Clock  
   - 칩, 시스템, 코어, 주변장치의 리셋과 클록 구성을 관리합니다.  
   - 기기 전체의 시작과 동작 속도를 맞추는 데 필요합니다.  
   - 클록과 리셋 제어가 노출되면 동작 방해가 쉬워질 수 있어 보호가 필요합니다.  

15. Low-Power Management  
   - 전원 도메인, PMU(전원 관리 장치), Sleep/Wake-up, RTC Boot(시간 유지용 회로 기반 시작) 등을 다룹니다.  
   - 배터리 수명을 늘리기 위한 핵심 기능입니다.  
   - 저전력 상태에서도 외부가 기기를 깨우거나 상태를 읽지 못하게 제한해야 합니다.  

16. UART Controller  
   - UART 직렬 통신을 위한 컨트롤러입니다.  
   - 외부 장치와 통신하거나 디버그 출력을 보낼 때 씁니다.  
   - 디버그 포트가 그대로 남아 있으면 내부 정보 유출 통로가 될 수 있습니다.  

17. SPI Controller  
   - SPI 방식의 주변장치나 메모리를 연결하는 통신 컨트롤러입니다.  
   - 빠른 속도가 필요한 저장장치나 센서 연결에 사용합니다.  
   - 외부 메모리를 쓴다면 데이터 보호도 같이 생각해야 합니다.  

18. I2C Controller  
   - I2C 방식의 주변장치 통신을 위한 컨트롤러입니다.  
   - 여러 센서나 주변기기를 적은 선으로 연결할 때 유용합니다.  
   - 통신선이 노출되면 주변기기 정보를 읽힐 수 있으니, 물리 보호와 접근 제한이 필요합니다.  

최종적으로 확인할 점은 이렇습니다.  
- 개발용 디버그 기능이 제품에서 그대로 열려 있지 않은지 확인해야 합니다.  
- UART, SWD/JTAG, SWO 같은 출력 경로에 민감한 정보가 남지 않게 해야 합니다.  
- GPIO와 통신선은 필요한 기능만 열고, 나머지는 막아야 합니다.  
- 저전력 기능을 쓰더라도 외부에서 쉽게 깨우거나 점검하지 못하게 해야 합니다.  
- 외부 메모리나 외부 통신을 쓰는 경우, 데이터가 밖으로 새지 않도록 함께 점검해야 합니다.  

현재 자료에서는 구체적인 회로 값이나 핀 번호까지는 확인하기 어렵습니다.  
원하시면 이 내용을 바탕으로 “스마트 도어락용 하드웨어 구성”처럼 기기별로 다시 쉽게 정리해드릴 수 있습니다.

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [x] CSV 파일 준비 및 데이터 확인 완료
- [x] CSV 데이터 탐색 및 통계 분석 완료
- [x] Supabase 연결 완료
- [x] CSV 데이터 업로드 완료
- [x] 직접 작성한 SQL 쿼리 테스트 완료 (최소 3개)
- [x] Text2SQL 함수 구현 및 프롬프트 수정 완료
- [x] 자연어 질문으로 SQL 생성 테스트 완료
- [x] 완전한 Text2SQL 시스템 (답변 생성) 테스트 완료
- [x] 최소 5개 이상의 다양한 질문으로 테스트 완료

---

## 추가 개선 아이디어

1. **프롬프트 개선**: SQL 생성 정확도 향상을 위한 예시 추가
2. **에러 핸들링**: SQL 오류 발생 시 재시도 로직 구현
3. **쿼리 검증**: 생성된 SQL이 안전한지 검증하는 로직 추가
4. **결과 포맷팅**: 테이블 형태로 결과 출력
5. **쿼리 히스토리**: 실행한 쿼리와 결과를 저장하여 재사용
6. **고급 SQL**: 서브쿼리, CTE, 윈도우 함수 활용